In [ ]:
import os
import glob
import pandas as pd
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
from astropy.convolution import Gaussian1DKernel
from astropy.convolution import convolve_fft
import sys
from scipy import stats
from scipy.optimize import curve_fit
import scipy.integrate
from sklearn.metrics import mean_squared_error
from numpy import log 
import scipy
from specutils import Spectrum1D
from specutils.manipulation import gaussian_smooth
from specutils.manipulation import FluxConservingResampler
from astropy import units as u
from sklearn.metrics import root_mean_squared_error
import warnings
from astropy.constants import c as c_vel


In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
def prep_stdfile(obs_id, HD, m_flag,stdspec_path):
    # FEROS files
    if obs_id == 'FEROS':
        name = HD + '_FEROS_norm.fits'

    # GOSSS files
    elif obs_id == 'GOSSS':
        name = HD + '_B2500.fits'

    # IACOB files
    elif obs_id == 'IACOB':
        name = HD + '*fits'

    # spectra from Hugues
    elif obs_id == 'concat':
        name = HD + '_concat.dat'

    # Hermes spectra
    else:
        obs_id = '0*' + obs_id

        # melchior spectrum
        if m_flag == 'y':
            name = obs_id + '_melchiors_spectrum*.fits'

        # not in melchior
        else:
            name = obs_id + '_HRF_OBJ_ext_CosmicsRemoved_log_merged_cf_norm*.fits'

    infile = glob.glob(stdspec_path + name)[0]

    return infile

In [ ]:
def read_HERMES(infile):
  #  print("%s: Input file is a HERMES file." % infile)
    header = fits.getheader(infile)
    bary_correc = header['BVCOR']
    # for Melchior-reduced files
    if not 'CRVAL1' in header:
 
        data = fits.open(infile)[1].data
        wave, flux = data['wave'], data['flux_norm']
        cdelt= None
 
    # for files with standard wavelegth array
    elif ((header['CTYPE1'] == 'WAVELENGTH') or (header['CTYPE1'] == 'AWAV')):
        print('-----here')
        flux = fits.getdata(infile)
        crval = header['CRVAL1']
        cdelt = header['CDELT1']
        naxis1 = header['NAXIS1']
        wave = crval + np.arange(0, naxis1) * cdelt
        print("lineal")
 
    # for files that are given in logarithmic wl array
    elif (header['CTYPE1'] == 'log(wavelength)'):
        flux = fits.getdata(infile)
        crval = header['CRVAL1']
        cdelt = header['CDELT1']
        naxis1 = header['NAXIS1']
        wave = np.exp(crval + np.arange(0, naxis1)*cdelt)
 
    else:
        print("Could not read in HERMES fits file - unknown file type.")
        sys.exit()
 
    return wave, flux,cdelt, bary_correc

In [ ]:
def get_instrument_by_obsid(obs_id):
    # Mapping obs_id to the corresponding instrument comment
    instruments = {
        'FEROS': 'FEROS',
        'GOSSS': 'GOSSS',
        'IACOB': 'IACOB',
        'concat': 'Hugues'
    }
    # Return the instrument comment, defaulting to 'Hermes spectra' if obs_id not found
    return instruments.get(obs_id, 'Hermes') 



In [ ]:
def ReadSpec_FEROS_fits(fits_file):

        """Function to read in fits file of high
        resolution spectra and then return wlen 
        and flux arrays"""
        
        # open fits file
        image=fits.open(fits_file)
        
        # retrieve header

        header=image[0].header
        
        # get object name
        star_name = header['OBJECT']
        
        # get coords
        ra_val = header['RA']
        dec_val = header['DEC']
        
                # check if we have CRPIX
        crpix1 = image[0].header.get("CRPIX1")
        
        if crpix1 is None:
            wlen, flux, err = image[1].data['WAVE'][0],image[1].data['FLUX'][0],image[1].data['ERR'][0]
            cdelt= None
        else :
            cdelt = header['CDELT1']
            #retrive flux
            flux=image[0].data
            # create wavelength array
            wlen=-header['CRPIX1']*cdelt + \
                      header['CRVAL1'] + \
                      np.arange(1,len(flux)+1)*cdelt
                  
      #  print ("Min. - Max. wlen: %i - %i \n" % (wlen[0], wlen[-1]))
        
        # file type to distinguish between        
        file_type = header['HIERARCH ESO PRO CATG']
        
        # date of observations
        mjd_obs = header['MJD-OBS']
        
                  
        # barycentric correction already applied for FEROS
        bary_correc = 0
                  
        image.close()
        
        # define star_name again with file type and obs date
        star_name = "%s_%s" % (star_name.replace(" ", ""), \
        file_type)
        
        return  wlen, flux,cdelt,header.get("SPEC_RES")

In [ ]:
def open_fits(filename, function):
    wave, flux, *_ = function(filename)
    return wave, flux

In [ ]:
def read_IACOB(filename):
    
        # open fits file
    image=fits.open(filename)
        
    # retrieve header

    header=image[0].header

    if ((header['CTYPE1'] == 'WAVELENGTH') or (header['CTYPE1'] == 'AWAV')):
        star_name = header['OBJECT']
        ra_val = header['RA']
        dec_val = header['DEC']
        crpix1 = header.get("CRPIX1")
        crval = header['CRVAL1']
        cdelt = header['CDELT1']
        naxis1 = header['NAXIS1']
        wave = crval + np.arange(0, naxis1) * cdelt
        flux = image[0].data
        return wave,flux,cdelt,header.get("I-RESOL"), header.get("BVCOR")

In [ ]:
# Función para convertir si es necesario a little-endian ("<")
def convert_to_little_endian(arr):
    # Si el array ya es little-endian, lo retornamos sin cambios
    if arr.dtype.byteorder == '<' or (arr.dtype.byteorder == '=' and sys.byteorder == 'little'):
        return arr
    # De lo contrario, cambiamos el orden de los bytes
    else:
        return arr.byteswap().newbyteorder('<')

In [ ]:
def normalize_spectra(wlen, flux,ivar,grado,range):
    flux_new = convert_to_little_endian(flux)
    lambda_new = convert_to_little_endian(wlen)
    ivar_new = convert_to_little_endian(ivar)
    if np.mean(lambda_new) <1000:
        lambda_new = 10 * lambda_new
    df_spec = pd.DataFrame({"lambda": lambda_new, "flux": flux_new,"ivar":ivar_new})
    df_spec = interpolar_ceros_flux_optimizado(df_spec)
    df_spec = df_spec.loc[(df_spec["lambda"]>=range[0])&(df_spec["lambda"]<=range[1])]
    df_spec["e_flux"] = np.sqrt(1/df_spec["ivar"])
    df_spec_complete = df_spec.copy()
    bins =  50
    df_spec["bins"] = pd.cut(df_spec["lambda"], bins=bins)
    df_spec['flux_std'] = df_spec.groupby("bins", observed=False)["flux"].transform('std')
    df_spec = df_spec.loc[df_spec["flux_std"] < df_spec["flux_std"].quantile(0.7)].reset_index(drop=True)
    SNR = (df_spec["flux"] *np.sqrt(df_spec["ivar"])).median()
    coeffs = np.polyfit(df_spec["lambda"], df_spec["flux"], grado)
    poly_vals = np.polyval(coeffs, df_spec_complete["lambda"])
    df_spec_complete["flux_norm"] = df_spec_complete["flux"] /poly_vals
    df_spec_complete["e_flux_norm"] = df_spec_complete["e_flux"] /poly_vals
    
    df_spec_complete.loc[(df_spec_complete["flux_norm"]<df_spec_complete['flux_norm'].quantile(0.8))&
            (df_spec_complete["flux_norm"]>df_spec_complete['flux_norm'].quantile(0.2)),"continuo"] = 1

    rms_flux = root_mean_squared_error(df_spec_complete.loc[
                                   df_spec_complete['continuo']==1]["flux_norm"], np.ones(len(df_spec_complete.loc[
                                   df_spec_complete['continuo']==1])))

    
    return df_spec_complete, SNR, rms_flux

In [ ]:
def agregar_ruido(flux,rms_lamost=None):
    df_spec_complete.loc[(df_spec_complete["flux_norm"]<df_spec_complete['flux_norm'].quantile(0.8))&
    (df_spec_complete["flux_norm"]>df_spec_complete['flux_norm'].quantile(0.2)),"continuo"] = 1  
    rms_flux = root_mean_squared_error(df_spec_complete.loc[
                               df_spec_complete['continuo']==1]["flux_norm"], np.ones(len(df_spec_complete.loc[
                               df_spec_complete['continuo']==1])))
    
    if rms_lamost ==notNone & (rms_lamost> rms_flux)&(rms_lamost!=0) :
        sigma_ruido = np.sqrt(rms_lamost**2-rms_flux**2)
        noise = np.random.normal(0, sigma_ruido, size=len(df_1300R["flux"]))
        df_1300R["flux"] =  df_1300R["flux"] + noise
        rms_flux = root_mean_squared_error(df_1300R.loc[
                               df_1300R['continuo']==1]["flux"], np.ones(len(df_1300R.loc[
                               df_1300R['continuo']==1])))
    return rms_flux


In [ ]:
def idl_tabulated(x, f, p=5):
    """Calcula la integral de los datos tabulados (x, f) usando el método de Newton-Cotes.
    
    Parámetros:
    - x: Array de NumPy con las coordenadas x de los datos.
    - f: Array de NumPy con las coordenadas f (función) de los datos.
    - p: Número de puntos a usar en cada segmento de Newton-Cotes.
    
    Retorna:
    - La integral calculada de los datos.
    """
    
    def newton_cotes_segmento(x_segmento, f_segmento):
        """Calcula la integral de un segmento usando el método de Newton-Cotes."""
        # Si el segmento tiene menos de 2 puntos, no se puede calcular una integral.
        if x_segmento.shape[0] < 2:
            return 0
        
        # Calcula los rangos normalizados para los pesos de Newton-Cotes.
        rangos_normalizados = (x_segmento.shape[0] - 1) * (x_segmento - x_segmento[0]) / (x_segmento[-1] - x_segmento[0])
        
        # Obtiene los pesos de Newton-Cotes para el segmento.
        pesos = scipy.integrate.newton_cotes(rangos_normalizados)[0]
        
        # Calcula la integral del segmento.
        integral_segmento = (x_segmento[-1] - x_segmento[0]) / (x_segmento.shape[0] - 1) * np.dot(pesos, f_segmento)
        
        return integral_segmento
    
    # Inicializa el resultado total de la integral.
    integral_total = 0
    
    # Itera a través de los segmentos de los datos.
    for idx in range(0, x.shape[0], p - 1):
        # Calcula la integral del segmento actual y la suma al total.
        integral_total += newton_cotes_segmento(x[idx:idx + p], f[idx:idx + p])
    
    # Retorna la integral total calculada.
    return integral_total


In [ ]:
def interpolar_ceros_flux_optimizado(df):
    """
    Optimiza la interpolación lineal de los valores de 'flux' que son cero en el DataFrame dado,
    utilizando operaciones vectorizadas en lugar de bucles.
    
    Parámetros:
    - df (pandas.DataFrame): DataFrame con las columnas 'wave' y 'flux'.
    
    Retorna:
    - df (pandas.DataFrame): DataFrame con los valores de 'flux' cero reemplazados por valores interpolados.
    """
    # Marcar los ceros con NaN para usar la interpolación de pandas
    df.loc[df['flux'] == 0, 'flux'] = np.nan

    # Interpolar linealmente los NaNs
    df["flux"] = df['flux'].interpolate(method='linear')

    return df

In [ ]:
def degrade_spec_to_Lamost1300(file_name,function,R,sigma_R1300_transformed,rms_lamost,save_spec=False):
    wave,flux = open_fits(file_name,function)
    
    spec1 = Spectrum1D(spectral_axis=wave * u.angstrom,
                       flux=flux*u.Jy)
    spec1_gsmooth = gaussian_smooth(spec1, stddev=sigma_R1300_transformed)
    
    fluxcon = FluxConservingResampler()
    #lamost sampling configuration
    naxis1 = 3909
    cdelt = 0.0001
    crval = 3.5682
    wave = 10**(crval + np.arange(0, naxis1)*cdelt)
    
    new_disp_grid = wave * u.Angstrom
    new_spec_fluxcon = fluxcon(spec1_gsmooth, new_disp_grid)

    df_1300R = pd.DataFrame({"lambda":new_spec_fluxcon.spectral_axis,"flux": new_spec_fluxcon.flux})
    df_1300R = df_1300R.loc[df_1300R["flux"].notna()].reset_index(drop=True)
    df_1300R = df_1300R.loc[(df_1300R["lambda"]>4000)&(df_1300R["lambda"]<6000)].reset_index(drop=True)


    df_1300R.loc[(df_1300R["flux"]<df_1300R['flux'].quantile(0.8))&
                (df_1300R["flux"]>df_1300R['flux'].quantile(0.2)),"continuo"] = 1
    
    rms_flux = root_mean_squared_error(df_1300R.loc[
                                       df_1300R['continuo']==1]["flux"], np.ones(len(df_1300R.loc[
                                       df_1300R['continuo']==1])))
    
    if (rms_lamost> rms_flux)&(rms_lamost!=0) :
        sigma_ruido = np.sqrt(rms_lamost**2-rms_flux**2)
        noise = np.random.normal(0, sigma_ruido, size=len(df_1300R["flux"]))
        df_1300R["flux"] =  df_1300R["flux"] + noise
        rms_flux = root_mean_squared_error(df_1300R.loc[
                                   df_1300R['continuo']==1]["flux"], np.ones(len(df_1300R.loc[
                                   df_1300R['continuo']==1])))
      #  print("LAMOST RMS is lower than standard star RMS; cannot increase noise to match LAMOST.")
    if save_spec==True:
        df_1300R.to_csv(f"spec_lamost_transformed_to_lamost/{file_name}_{np.round(rms_lamost)}.csv")
        
    
    return df_1300R,rms_flux

In [ ]:
warnings.filterwarnings('ignore', message="nan_treatment='interpolate', however, NaN values detected post convolution")

In [ ]:
path = "/home/nicolas/nico/Data/Masivas/Data_OB_stars/StandardMasivas/"

In [ ]:
df = pd.read_csv(f'{path}Spectral_typing_Didelon.csv')
stdspec_path = path+'std_specs/'
df = df.loc[df["excl"] == "n"].reset_index(drop=True)
df['instrument'] = df['ID'].apply(get_instrument_by_obsid)

In [ ]:

df["sp"] = df["SpT"].str[0:2]

df.loc[df["SpT"].str[0]=="O","sp"] = "O"


In [ ]:
df = df.loc[df["LC"]=="V"].reset_index(drop=True)

In [ ]:
df["fits"] = df.apply(lambda row: prep_stdfile(row['ID'], row['HD_num'], row['melchior'],stdspec_path), axis=1)

In [ ]:
function_map = {
    'IACOB': read_IACOB,
    'FEROS': ReadSpec_FEROS_fits,
    'Hermes': read_HERMES
}

# Asignar las funciones a una nueva columna en el DataFrame
df['function'] = df['instrument'].map(function_map)

In [ ]:
#hermes todos logaritmico
hermes_cdelt = []
hermes_rv = []
for infile in df.loc[df["instrument"]=="Hermes"]["fits"].reset_index(drop=True).values:
    wave,flux,cdelt,rv =read_HERMES(infile)
    hermes_rv.append(rv)
    diferencias_abs = np.absolute(wave - 4471)
    # Encuentra el índice donde el valor absoluto es mínimo
    index_min = np.argmin(diferencias_abs)
    delta_mas = wave[index_min+1] - wave[index_min]
    delta_menos = wave[index_min] - wave[index_min-1]
    if delta_mas<=delta_menos:
        hermes_cdelt.append(delta_mas)
    else:
        hermes_cdelt.append(delta_menos)
#no encontre la resolucion en el header        
df.loc[df["instrument"]=="Hermes","cdelt"] = hermes_cdelt
df.loc[df["instrument"]=="Hermes","R"] = 85000
df.loc[df["instrument"]=="Hermes","rv"] = hermes_rv

In [ ]:
#IACOB ninguno logaritmico
IACOB_cdelt = []
IACOB_res = []
IACOB_rv = []

for infile in df.loc[df["instrument"]=="IACOB"]["fits"].reset_index(drop=True).values:
    wave,flux,cdelt,res,rv =read_IACOB(infile)
    IACOB_cdelt.append(cdelt)
    IACOB_res.append(res)
    IACOB_rv.append(rv)
df.loc[df["instrument"]=="IACOB","R"] = IACOB_res
df.loc[df["instrument"]=="IACOB","rv"] = IACOB_rv
df.loc[df["instrument"]=="IACOB","cdelt"] = IACOB_cdelt

In [ ]:
df["rv"]= pd.to_numeric(df["rv"])

In [ ]:
#Feros Ninguno logaritmico
FEROS_cdelt = []
FEROS_res = []
for infile in df.loc[df["instrument"]=="FEROS"]["fits"].reset_index(drop=True).values:
    wave,flux,cdelt,res = ReadSpec_FEROS_fits(infile)
    FEROS_cdelt.append(cdelt)
    FEROS_res.append(res)
df.loc[df["instrument"]=="FEROS","R"] = FEROS_res
df.loc[df["instrument"]=="FEROS","cdelt"] = FEROS_cdelt

In [ ]:
FWHM_binning = df["cdelt"]
FWHM_R85000 = 4471/df["R"]
FWHM_R1300 = 4471/1300
FWHM_binning_to_R85000 = FWHM_R85000/FWHM_binning
FWHM_R85000_to_R1300 = FWHM_R1300/FWHM_R85000
FWHM_R1300_transformed = FWHM_R85000_to_R1300*FWHM_binning_to_R85000
sigma_R1300_transformed = FWHM_R1300_transformed/2.355
df["sigma_R1300_transformed"] = sigma_R1300_transformed

In [ ]:
path_plots = f"{path}/plots"

In [ ]:
#Lamost high SNR
image = fits.open("/home/nicolas/nico/Data/Masivas/Data_OB_stars/spectra/LAMOST_low_resolution/sp/spec-56345-GAC079N41V2_sp15-192.fits")
flux_l,ivar,wave_l,andmask,ormask,norm = image[1].data[0]
df_lamost,snr,rms_lamost = normalize_spectra(wave_l, flux_l, ivar, 1, [4400,4700])

df_review = df.loc[df["SpT"]=="O7"].sample(1).reset_index(drop=True)
wave, flux, *_ = df_review["function"][0](df_review["fits"][0])
df_1300,rms = degrade_spec_to_Lamost1300(df_review["fits"][0],
                    df_review["function"][0],
                    df_review["R"][0],
                    df_review["sigma_R1300_transformed"][0],rms_lamost)
rv = df_review["rv"].values
df_1300["lambda"] = df_1300["lambda"] + (rv*u.km/u.s*df_1300["lambda"])/c_vel
wave = wave + (rv*u.km/u.s*wave)/c_vel
f, ax = plt.subplots()  
R = df_review["R"].values[0]
ax.plot(wave, flux, label=f"{df[df['SpT'] == 'O7']['Name'].values[0]} [R ~{R}]", c="black", alpha=0.4)
ax.plot(df_1300["lambda"], df_1300["flux"], label=f"{df[df['SpT'] == 'O7']['Name'].values[0]} [R ~1300]", c="black")
ax.plot(df_lamost["lambda"], df_lamost["flux_norm"], c="red", label="HD277878 [Lamost spectra R ~1300]")
plt.ylim(0.6, 1.07)
plt.xlim(4450, 4700)
plt.xlabel(r"Lambda ($\AA$)", fontsize=15)
plt.ylabel("Normalized Flux",fontsize=15)
plt.legend(fontsize=11)
plt.savefig(f"{path_plots}/Standard_degraded.pdf", format='pdf', bbox_inches='tight')

In [ ]:
# Ruta al directorio que contiene los archivos .fits
directory = "/home/nicolas/nico/Data/Masivas/Data_OB_stars/spectra/LAMOST_low_resolution/sp"

# Lista para almacenar los valores de rms_lamost
rms_values = []

# Itera sobre todos los archivos en el directorio
for filename in os.listdir(directory):
    if filename.endswith(".fits"):
        file_path = os.path.join(directory, filename)
        image = fits.open(file_path)
        flux_l, ivar, wave_l, andmask, ormask, norm = image[1].data[0]
        snrg = image[0].header["SNRG"]
        
        # Asumiendo que normalize_spectra es una función definida en otro lugar y disponible aquí
        df_lamost, snr, rms_lamost = normalize_spectra(wave_l, flux_l, ivar, 1, [4000, 5000])
        if snrg >50:
            # Agregar el valor de rms_lamost a la lista
            rms_values.append(rms_lamost)

In [ ]:
from pandarallel import pandarallel

# Initialize pandarallel with the progress bar.
pandarallel.initialize(progress_bar=True)

In [ ]:
df["rms_nativo"] = df.parallel_apply(lambda row: degrade_spec_to_Lamost1300(row["fits"],
                                                                   row["function"],
                                                                   row["R"],
                                                                   row["sigma_R1300_transformed"],
                                                                   0)[1], axis=1)

np.random.seed(1)  # Set the random seed for reproducibility
selection = np.random.choice(rms_values, len(df))
df["Lamost_rms"] = selection
df["rms_transformed_to_lamost"] = df.parallel_apply(lambda row: degrade_spec_to_Lamost1300(row["fits"],
                                                                   row["function"],
                                                                   row["R"],
                                                                   row["sigma_R1300_transformed"],
                                                                   row["Lamost_rms"])[1], axis=1)


In [ ]:
df.loc[df["rv"].isna(),"rv"] = 0 

In [ ]:
#np.random.seed(1)  # Set the random seed for reproducibility
#selection = np.random.choice(rms_values, len(df))
#df["Lamost_rms"] = selection
#df.parallel_apply(lambda row: degrade_spec_to_Lamost1300(row["fits"],
#                                                                   row["function"],
#                                                                   row["R"],
#                                                                   row["sigma_R1300_transformed"],
#                                                                   row["Lamost_rms"],
#                                                                    row["HD_nu"],save_spec=True), axis=1)

In [ ]:
from scipy import stats
import seaborn as sns

In [ ]:

sns.histplot(x=rms_values,bins=10,stat="density",log_scale=True, label="RMS LAMOST")
sns.histplot(data=df,x="rms_nativo",bins=10,stat="density",log_scale=True, label="RMS Standards Native")
w_dist = stats.wasserstein_distance(rms_values, df["rms_nativo"])
w_dist_round = np.round(w_dist, 3)

# Título con el valor de la distancia
plt.title(f'Wasserstein: {w_dist_round}')
plt.legend()

In [ ]:
sns.histplot(x=rms_values,bins=10,stat="density",log_scale=True, label="RMS LAMOST")
sns.histplot(data=df,x="rms_transformed_to_lamost",bins=10,stat="density", label="RMS Standards R LAMOST")
w_dist = stats.wasserstein_distance(rms_values, df["rms_transformed_to_lamost"])
w_dist_round = np.round(w_dist, 3)

# Título con el valor de la distancia
plt.title(f'Wasserstein: {w_dist_round}')
plt.legend()

# Detectar Linea [Siguiendo del paper]
##### usando continuos del paper anterior, seleccionan solamente una peque;a parte del continuo y por eso no puedo aplicar 3 continuos quitando los que tienen una mayor desviacion estandar, porque puedo gran parte del continuo.

In [ ]:
df["sp_Monsalves"] = df["SpT"]

df.loc[df["SpT"].str[0]=="O","sp_Monsalves"] = "O"

df.loc[(df["SpT"].str[0:2]=="B0"),"sp_Monsalves"] = "B0"

df.loc[(df["SpT"].str[0:2]=="B1")|
        (df["SpT"].str[0:2]=="B2"),"sp_Monsalves"] = "B1/B2"

df.loc[(df["SpT"].str[0:2]=="B3")|
        (df["SpT"].str[0:2]=="B4")|
        (df["SpT"].str[0:2]=="B5"),"sp_Monsalves"] = "B3/B5"

df.loc[(df["SpT"].str[0:2]=="B6")|
        (df["SpT"].str[0:2]=="B7")|
        (df["SpT"].str[0:2]=="B8"),"sp_Monsalves"] = "B6/B8"

df.loc[(df["SpT"].str[0:2]=="B9")|
        (df["SpT"].str[0:1]=="A"),"sp_Monsalves"] = "B9/A"

In [ ]:
lines = {
    "He I 4009": {
        "spectral_line": (4004, 4016),
        "continuum_blue": (3990, 4002),
        "continuum_red": (4035, 4060)
    },
    "He I+He II 4026": {
        "spectral_line": (4017, 4034),
        "continuum_blue": (3990, 4002),
        "continuum_red": (4035, 4060)
    },
    "Si IV 4088": {
        "spectral_line": (4084, 4091),
        "continuum_blue": (4055, 4075),
        "continuum_red": (4150, 4170)
    },
    "H δ 4100": {
        "spectral_line": (4095, 4110),
        "continuum_blue": (4055, 4075),
        "continuum_red": (4150, 4170)
    },
    "Si IV 4116": {
        "spectral_line": (4113, 4118),
        "continuum_blue": (4055, 4075),
        "continuum_red": (4150, 4170)
    },
    "He I 4121": {
        "spectral_line": (4118, 4125),
        "continuum_blue": (4035, 4060),
        "continuum_red": (4150, 4170)
    },
    "Si II 4130": {
        "spectral_line": (4125, 4135),
        "continuum_blue": (4035, 4060),
        "continuum_red": (4150, 4190)
    },
    "He I 4144": {
        "spectral_line": (4140, 4150),
        "continuum_blue": (4035, 4060),
        "continuum_red": (4150, 4170)
    },
    "He II 4200": {
        "spectral_line": (4190, 4207),
        "continuum_blue": (4150, 4190),
        "continuum_red": (4245, 4260)
    },
    "Fe II 4233": {
        "spectral_line": (4229, 4237),
        "continuum_blue": (4205, 4225),
        "continuum_red": (4238, 4260)
    },
    "He I 4387": {
        "spectral_line": (4382, 4392),
        "continuum_blue": (4365, 4380),
        "continuum_red": (4398, 4415)
    },
    "O II 4416": {
        "spectral_line": (4412, 4421),
        "continuum_blue": (4398, 4411),
        "continuum_red": (4440, 4460)
    },
    "He I 4471": {
        "spectral_line": (4462, 4477),
        "continuum_blue": (4440, 4460),
        "continuum_red": (4495, 4535)
    },
    "Mg II 4481": {
        "spectral_line": (4477, 4488),
        "continuum_blue": (4440, 4460),
        "continuum_red": (4490, 4505)
    },
    "He II 4541": {
        "spectral_line": (4537, 4547),
        "continuum_blue": (4510, 4535),
        "continuum_red": (4590, 4620)
    },
    "Si III 4553": {
        "spectral_line": (4548, 4558),
        "continuum_blue": (4485, 4507),
        "continuum_red": (4600, 4620)
    },
    "O II+C III 4645": {
        "spectral_line": (4635, 4655),
        "continuum_blue": (4600, 4625),
        "continuum_red": (4660, 4685)
    },
    "He II 4686": {
        "spectral_line": (4679, 4692),
        "continuum_blue": (4660, 4670),
        "continuum_red": (4730, 4745)
    }
}

for key, values in lines.items():
    lines[key]['central_lambda'] = float(key.split()[-1])

In [ ]:
def Detect_line_static_and_var(filename,function,R,rv,sigma_R1300_transformed,sp,rms_lamost,blue_cont,red_cont,spec_line,lambda_peak,threshold,ax=None,plot=False):
    spectra,rms = degrade_spec_to_Lamost1300(filename,
                    function,
                    R,
                    sigma_R1300_transformed,rms_lamost)

    spectra = spectra.loc[(spectra["lambda"]>blue_cont[0])&
                                  (spectra["lambda"]<red_cont[1])]

    spectra["lambda"] = spectra["lambda"] + (rv*u.km/u.s*spectra["lambda"])/c_vel
    df_blue_red_region = spectra.loc[
        ((spectra["lambda"] > blue_cont[0]) & (spectra["lambda"] < blue_cont[1])) |
        ((spectra["lambda"] > red_cont[0]) & (spectra["lambda"] < red_cont[1]))
    ]
            
    sp_out_line = spectra.loc[
        ((spectra["lambda"] > blue_cont[0]) & (spectra["lambda"] < spec_line[0])) |
        ((spectra["lambda"] > spec_line[1]) & (spectra["lambda"] < red_cont[1]))
    ]


    slope_1, intercept_1, r_value_1, p_value_1, std_err_1 = stats.linregress(df_blue_red_region["lambda"], df_blue_red_region["flux"])
    fit_1 = slope_1 * sp_out_line["lambda"] + intercept_1 
    sp_out_line["flux-cont_1"] = sp_out_line["flux"] - fit_1
    dispersion_1 = sp_out_line["flux-cont_1"].std()
    df_top_bot = sp_out_line.loc[np.absolute(sp_out_line["flux-cont_1"])<dispersion_1]


    slope_3, intercept_3, r_value_3, p_value_3, std_err_3 = stats.linregress(df_top_bot["lambda"], df_top_bot["flux"])
    fit_3 = slope_3 * sp_out_line["lambda"] + intercept_3
    dispersion_third_fit = np.std(np.absolute(df_top_bot["flux"] - fit_3))

    spectra["third_cont"] = np.polyval([slope_3,intercept_3], spectra["lambda"])
    spectra["flux_norm_line"] = spectra["flux"] /spectra["third_cont"]
    df_line = spectra.loc[(spectra["lambda"]>spec_line[0])&
                      (spectra["lambda"]<spec_line[1])].reset_index(drop=True)


    df_line["flux-cont"] = np.absolute(df_line["flux"] - df_line["third_cont"])
    abs_flux_cont_q = df_line.loc[(df_line["lambda"]>lambda_peak-3)&
                                    (df_line["lambda"]<lambda_peak+3)]["flux-cont"].quantile(0.95)
    
    df_line_over_q = df_line.loc[(df_line["flux-cont"]>abs_flux_cont_q)&
                                    (df_line["lambda"]>lambda_peak-3)&
                                    (df_line["lambda"]<lambda_peak+3)].reset_index(drop=True)
    print(df_line_over_q)
    if (df_line_over_q["flux-cont"] >  threshold*dispersion_third_fit).any():
        detection = "E"
        if (df_line_over_q["flux"] > df_line_over_q["third_cont"]).any():
            line_type = "ems"
        else:
            line_type = "abs"
    else:
        line_type="U"
    upper_limit = df_line["third_cont"] + threshold*dispersion_third_fit
    lower_limit  = df_line["third_cont"] - threshold*dispersion_third_fit        
    if plot==True:        
        ax.plot(spectra["lambda"], spectra["flux"], "o-", c="grey", alpha=0.5)
        ax.plot(df_top_bot["lambda"], df_top_bot["flux"], ".", c="blue")
        ax.plot(df_line["lambda"],df_line["flux"], "o-",color='black')
    
        ax.plot(sp_out_line["lambda"], fit_1, linestyle='dotted', c="red")
        ax.plot(sp_out_line["lambda"], fit_3, linestyle='--', c="blue")
        ax.fill_between(df_line["lambda"], upper_limit, lower_limit, color='orange', alpha=0.5, label=f'{threshold} Sigma')
        ax.axvline(x=lambda_peak, color='purple', linestyle='--', label='Vertical Line')
    
        ax.set_title(f"{line_type}[{sp}]")            
        ax.plot(df_line_over_q["lambda"], df_line_over_q["flux"], "o-", c="purple", label="quantile 0.95")
        ax.legend()
        ax.set_ylim(spectra["flux"].min(), 1.03)
        return 
    else:
        coeffs = (slope_3,intercept_3)
        return spectra,coeffs, line_type

def Detect_line_static(filename,function,R,rv,sigma_R1300_transformed,sp,rms_lamost,blue_cont,red_cont,spec_line,lambda_peak,threshold,ax=None,plot=False):
    spectra,rms = degrade_spec_to_Lamost1300(filename,
                    function,
                    R,
                    sigma_R1300_transformed,rms_lamost)

    spectra = spectra.loc[(spectra["lambda"]>blue_cont[0])&
                                  (spectra["lambda"]<red_cont[1])]

    spectra["lambda"] = spectra["lambda"] + (rv*u.km/u.s*spectra["lambda"])/c_vel
    df_blue_red_region = spectra.loc[
        ((spectra["lambda"] > blue_cont[0]) & (spectra["lambda"] < blue_cont[1])) |
        ((spectra["lambda"] > red_cont[0]) & (spectra["lambda"] < red_cont[1]))
    ]
            
    sp_out_line = spectra.loc[
        ((spectra["lambda"] > blue_cont[0]) & (spectra["lambda"] < spec_line[0])) |
        ((spectra["lambda"] > spec_line[1]) & (spectra["lambda"] < red_cont[1]))
    ]

    #no esta bien el sufijo, pero mantengo el _3 para no tener problemas con variables
    slope_3, intercept_3, r_value_3, p_value_3, std_err_3 = stats.linregress(df_blue_red_region["lambda"], df_blue_red_region["flux"])
    fit_3 = slope_3 * sp_out_line["lambda"] + intercept_3
    dispersion_third_fit = np.std(np.absolute(df_blue_red_region["flux"] - fit_3))


    spectra["third_cont"] = np.polyval([slope_3,intercept_3], spectra["lambda"])
    spectra["flux_norm_line"] = spectra["flux"] /spectra["third_cont"]
    df_line = spectra.loc[(spectra["lambda"]>spec_line[0])&
                      (spectra["lambda"]<spec_line[1])].reset_index(drop=True)


    df_line["flux-cont"] = np.absolute(df_line["flux"] - df_line["third_cont"])
    abs_flux_cont_q = df_line.loc[(df_line["lambda"]>lambda_peak-3)&
                                    (df_line["lambda"]<lambda_peak+3)]["flux-cont"].quantile(0.95)
    
    df_line_over_q = df_line.loc[(df_line["flux-cont"]>abs_flux_cont_q)&
                                    (df_line["lambda"]>lambda_peak-3)&
                                    (df_line["lambda"]<lambda_peak+3)].reset_index(drop=True)
    print(df_line_over_q)
    if (df_line_over_q["flux-cont"] >  threshold*dispersion_third_fit).any():
        detection = "E"
        if (df_line_over_q["flux"] > df_line_over_q["third_cont"]).any():
            line_type = "ems"
        else:
            line_type = "abs"
    else:
        line_type="U"
    upper_limit = df_line["third_cont"] + threshold*dispersion_third_fit
    lower_limit  = df_line["third_cont"] - threshold*dispersion_third_fit        
    if plot==True:        
        ax.plot(spectra["lambda"], spectra["flux"], "o-", c="grey", alpha=0.5)
        ax.plot(df_line["lambda"],df_line["flux"], "o-",color='black')
        ax.plot(sp_out_line["lambda"], fit_3, linestyle='dotted', c="red")
        ax.fill_between(df_line["lambda"], upper_limit, lower_limit, color='orange', alpha=0.5, label=f'{threshold} Sigma')
        ax.axvline(x=lambda_peak, color='purple', linestyle='--', label='Vertical Line')
    
        ax.set_title(f"{line_type}[{sp}]")            
        ax.plot(df_line_over_q["lambda"], df_line_over_q["flux"], "o-", c="purple", label="quantile 0.95")
        ax.legend()
        ax.set_ylim(spectra["flux"].min(), 1.03)
        return 
    else:
        coeffs = (slope_3,intercept_3)
        return spectra,coeffs, line_type

def gauss(x, A, x0, sigma,b):
    return A * np.exp(-(x - x0)**2 / (2 * sigma**2)) + b

def stimate_parameters_line_type(lambda_peak,cota_mean,df_continuo,line_type):
    if line_type=="abs":
        A_1 = df_continuo.loc[(df_continuo["lambda"]> lambda_peak -cota_mean) & (df_continuo["lambda"] <lambda_peak + cota_mean)]["flux_norm_line"].min() -  df_continuo["flux_norm_line"].mean()
        amplitude_1 = -(1-A_1)
        A_lim_inf = -np.inf
        A_lim_sup = 0
    if line_type=="ems":
        A_1 = df_continuo.loc[(df_continuo["lambda"]> lambda_peak -cota_mean) & (df_continuo["lambda"] <lambda_peak + cota_mean)]["flux_norm_line"].max() -  df_continuo["flux_norm_line"].mean()
        amplitude_1 = A_1
        A_lim_inf = 0
        A_lim_sup = np.inf  
    return amplitude_1,A_lim_sup,A_lim_inf



def gauss_core_emission(x, A1, x01, sigma1, b1, A2, sigma2):
    # Primera gaussiana (puede ser emisión o absorción dependiendo del signo de A1)
    gaussian_first = gauss(x, A1, x01, sigma1, b1)
    # Segunda gaussiana (siempre en emisión y centrada en el pico de la primera)
    gaussian_second = gauss(x, A2, x01, sigma2, 0)  # b2 es cero para que solo ajuste el pico
    # Combina las dos gaussianas
    return gaussian_first + gaussian_second

def dos_gaussianas(x, A1, x01, sigma1, A2, x02, sigma2, b):
    return gauss(x, A1, x01, sigma1,b/2) + gauss(x, A2, x02, sigma2,b/2)



def calculate_bic(n, mse, num_params):
    bic = n * log(mse) + num_params * log(n)
    return bic


def Detect_line_bayo2011(filename,function,R,rv,sigma_R1300_transformed,sp,rms_lamost,lambda_peak,threshold,ax=None):
    try:
        spectra,rms = degrade_spec_to_Lamost1300(filename,
                        function,
                        R,
                        sigma_R1300_transformed,rms_lamost)
            
        spectra["lambda"] = spectra["lambda"] + (rv*u.km/u.s*spectra["lambda"])/c_vel
        
        # Filtrar los datos antes y después del intervalo
        blue = spectra.loc[spectra["lambda"] < (lambda_peak)].tail(100)
        red = spectra.loc[spectra["lambda"] > (lambda_peak)].head(100)
        
        # Combinar los resultados
        df_blue_red_region = pd.concat([blue, red]).reset_index(drop=True)
        
        
        slope_1, intercept_1, r_value_1, p_value_1, std_err_1 = stats.linregress(df_blue_red_region["lambda"], df_blue_red_region["flux"])
        fit_1 = slope_1 * df_blue_red_region["lambda"] + intercept_1 
        
        coeffs = (slope_1,intercept_1)
        
        df_blue_red_region["first_cont"] = np.polyval([slope_1,intercept_1], df_blue_red_region["lambda"])
        df_blue_red_region["flux_norm_line"] = df_blue_red_region["flux"] /df_blue_red_region["first_cont"]
        
        spec_line = df_blue_red_region.reset_index(drop=True)["lambda"].iloc[[0, -1]].values
        
        df_blue_red_region["flux-cont"] = np.absolute(df_blue_red_region["flux"] - df_blue_red_region["first_cont"])
        
        abs_flux_cont_q = df_blue_red_region.loc[(df_blue_red_region["lambda"]>lambda_peak-3)&
                                        (df_blue_red_region["lambda"]<lambda_peak+3)]["flux-cont"].quantile(0.95)
        
        df_line_over_q = df_blue_red_region.loc[(df_blue_red_region["flux-cont"]>abs_flux_cont_q)&
                                        (df_blue_red_region["lambda"]>lambda_peak-3)&
                                        (df_blue_red_region["lambda"]<lambda_peak+3)].reset_index(drop=True)
        
        if (df_line_over_q["flux"] > df_line_over_q["first_cont"]).any():
            line_type = "ems"
        else:
            line_type = "abs"
        
        p_predicted,bic,EW = gauss_simple_fit(df_blue_red_region,lambda_peak,spec_line,line_type,coeffs)
        
        sigma = p_predicted[2]
        
        x_coords_first = [p_predicted[1] - 3 * p_predicted[2], p_predicted[1] + 3 * p_predicted[2]]
        y_coords_first = [1+(p_predicted[0]* (1 / np.sqrt(2))),1+(p_predicted[0]* (1 / np.sqrt(2)))]
        
        df_close_edge = df_blue_red_region.loc[(df_blue_red_region["lambda"]<=lambda_peak-10*sigma)|
                                        (df_blue_red_region["lambda"]>=lambda_peak+10*sigma)]
        
        slope_2, intercept_2, r_value_2, p_value_2, std_err_2 = stats.linregress(df_close_edge["lambda"], df_close_edge["flux"])
        fit_2 = slope_2 * df_close_edge["lambda"] + intercept_2 
        
        df_blue_red_region["flux-cont_2"] = df_blue_red_region["flux"] - fit_2
        dispersion_2 = df_blue_red_region["flux-cont_2"].std()
        df_blue_red_region_dispersion = df_blue_red_region.loc[np.absolute(df_blue_red_region["flux-cont_2"])<dispersion_2]
        
        slope_3, intercept_3, r_value_3, p_value_3, std_err_3 = stats.linregress(df_blue_red_region_dispersion["lambda"], df_blue_red_region_dispersion["flux"])
        fit_3 = slope_3 * df_blue_red_region["lambda"] + intercept_3
        
        df_blue_red_region["third_cont"] = np.polyval([slope_3,intercept_3], df_blue_red_region["lambda"])
        df_blue_red_region["flux_norm_line"] = df_blue_red_region["flux"] /df_blue_red_region["third_cont"]
        
        coeffs = (slope_3,intercept_3)
        spec_line = df_blue_red_region_dispersion.reset_index(drop=True)["lambda"].iloc[[0, -1]].values
        p_predicted,bic,EW = gauss_simple_fit(df_blue_red_region,lambda_peak,spec_line,line_type,coeffs)
        
        x_coords_second = [p_predicted[1] - p_predicted[2], p_predicted[1] + p_predicted[2]]
        y_coords_second = [1+(p_predicted[0]* (1 / np.sqrt(2))),1+(p_predicted[0]* (1 / np.sqrt(2)))]
        
        
        offsets = [3, 4, 5]
        results = {}
        
        for offset in offsets:
            lambda_1 = lambda_peak - offset * p_predicted[2]
            lambda_2 = lambda_peak + offset * p_predicted[2]
            F_1 = df_blue_red_region.loc[(df_blue_red_region['lambda'] - lambda_1).abs().idxmin(), 'flux']
            F_2 = df_blue_red_region.loc[(df_blue_red_region['lambda'] - lambda_2).abs().idxmin(), 'flux']
            results[offset] = {
                'lambda_1': lambda_1,
                'lambda_2': lambda_2,
                'F_1': F_1,
                'F_2': F_2
            }
        
            df_line = df_blue_red_region[
                ((df_blue_red_region['lambda'] >= lambda_1) & (df_blue_red_region['lambda'] <= lambda_2)) 
            ]
        
        dispersion_third_fit = np.std(np.absolute(df_blue_red_region_dispersion["flux"] - fit_3))
        
        df_line = df_blue_red_region[
                ((df_blue_red_region['lambda'] >= lambda_1) & (df_blue_red_region['lambda'] <= lambda_2)) 
            ]
        df_line["third_cont"] =  np.polyval([slope_3,intercept_3], df_line["lambda"])
        
        df_line["flux-cont"] = np.absolute(df_line["flux"] - df_line["third_cont"])
        abs_flux_cont_q = df_line.loc[(df_line["lambda"]>lambda_peak-3)&
                                        (df_line["lambda"]<lambda_peak+3)]["flux-cont"].quantile(0.95)
        
        df_line_over_q = df_line.loc[(df_line["flux-cont"]>abs_flux_cont_q)&
                                        (df_line["lambda"]>lambda_peak-3)&
                                        (df_line["lambda"]<lambda_peak+3)].reset_index(drop=True)
        print(df_line_over_q)
        if (df_line_over_q["flux-cont"] >  threshold*dispersion_third_fit).any():
            detection = "E"
            if (df_line_over_q["flux"] > df_line_over_q["third_cont"]).any():
                line_type = "ems"
            else:
                line_type = "abs"
        else:
            line_type="U"
        upper_limit = df_line["third_cont"] + threshold*dispersion_third_fit
        lower_limit  = df_line["third_cont"] - threshold*dispersion_third_fit  
        
        ax.plot(df_blue_red_region["lambda"],df_blue_red_region["flux"],"o-",c="grey")
        ax.plot(df_line["lambda"],df_line["flux"],"o-",c="black")
        ax.plot(df_blue_red_region["lambda"],fit_3,"blue",linestyle="dashed")
        ax.plot(x_coords_second, y_coords_second, color='blue')
        ax.plot(x_coords_first, y_coords_first, color='red')
        ax.plot(df_blue_red_region["lambda"],fit_1,"red", linestyle="dotted")
        ax.plot(df_blue_red_region_dispersion["lambda"],df_blue_red_region_dispersion["flux"],"k.",c="blue")
        
        line_styles = {
            3: 'solid',
            4: 'dotted',
            5: 'dashed'
        }
        
        for offset in offsets:
            lambda_1 = results[offset]['lambda_1']
            lambda_2 = results[offset]['lambda_2']
            F_1 = results[offset]['F_1']
            F_2 = results[offset]['F_2']
            ax.plot([lambda_1, lambda_2], [F_1, F_2], label=f'Offset {offset}', color='lime', linestyle=line_styles[offset])
            ax.axvline(lambda_1, color='lime', linestyle=line_styles[offset])
            ax.axvline(lambda_2, color='lime', linestyle=line_styles[offset])
        ax.fill_between(df_line["lambda"], upper_limit, lower_limit, color='orange', alpha=0.5, label=f'{threshold} Sigma')
        ax.set_title(f"{line_type}[{sp}]")            

        ax.legend()
        return p_predicted,line_type
    except:
        ax.plot(df_blue_red_region["lambda"],df_blue_red_region["flux"],"o-",c="grey")
        ax.set_title(f"U[{sp}]")

In [ ]:
def ajuste_gaussiano(df_continuo,lambda_peak_1,lambda_peak_2,line_type,coeffs,spec_line,cota_mean,R,sp,plot=False,ax=None):
    # Extraer datos
    x_datos = df_continuo["lambda"].values
    y_datos = df_continuo["flux_norm_line"].values
    # Valores iniciales para los parámetros del ajuste
    sigma = lambda_peak_1/(R*2.634)
    b = df_continuo["flux_norm_line"].quantile(0.9)

    ##AJUSTE UNA GAUSSIANA

    amplitude_1,A_lim_sup_1,A_lim_inf_1 = stimate_parameters_line_type(lambda_peak_1,3,df_continuo,line_type)
    p0_simple = [amplitude_1, lambda_peak_1, sigma, b]
    bounds_simple = ([A_lim_inf_1, lambda_peak_1-cota_mean, sigma, 0], [A_lim_sup_1, lambda_peak_1+cota_mean,np.inf, 1.5])
    # Ajuste de una sola gaussiana
    p_opt_simple, p_cov_simple = curve_fit(gauss, x_datos, y_datos, p0=p0_simple, bounds=bounds_simple, max_nfev=10000)
    df_linea = df_continuo.loc[(df_continuo["lambda"]> p_opt_simple[1] - 4*p_opt_simple[2]) & (df_continuo["lambda"]< p_opt_simple[1] + 4*p_opt_simple[2])]
    x_linea,y_linea = df_linea["lambda"].values,df_linea["flux"].values
    mse_simple = mean_squared_error(y_linea,  gauss(x_linea, *p_opt_simple))
    bic_simple = calculate_bic(len(y_linea), mse_simple, len(p_opt_simple))
    if bic_simple >-50:
        return "U"
    
    # AJUSTE DOS GAUSSIANAS
    if lambda_peak_2 != None :
        delta_lambda = p_opt_simple[1] - lambda_peak_1
        lambda_peak_2 = lambda_peak_2+delta_lambda
        amplitude_2,A_lim_sup_2,A_lim_inf_2 = stimate_parameters_line_type(lambda_peak_2,3,df_continuo,line_type) 
        p0_doble = [amplitude_1,lambda_peak_1, sigma, amplitude_2, lambda_peak_2, sigma, b]
        bounds_doble = ([-np.inf, lambda_peak_1-cota_mean, sigma, -np.inf, lambda_peak_2-cota_mean, sigma, 0], 
                        [0, lambda_peak_1 + cota_mean, np.inf, 0, lambda_peak_2+cota_mean, np.inf,1.5])
        p_opt_doble, p_cov_doble = curve_fit(dos_gaussianas, x_datos, y_datos, p0=p0_doble, bounds=bounds_doble, max_nfev=10000)
        mse_doble = mean_squared_error(y_linea,  dos_gaussianas(x_linea, *p_opt_doble))
        bic_doble = calculate_bic(len(y_linea), mse_doble, len(p_opt_doble))
    else:
        bic_doble = 999
        p_opt_doble = [999,999,999]

    #AJUSTE CORE EMISION

    p0_core_emission = [p_opt_simple[0] , p_opt_simple[1], p_opt_simple[2] , p_opt_simple[-1],-amplitude_1/2,sigma]

    bounds_core_emission =     ([-np.inf, lambda_peak_1-cota_mean, sigma , 0, 0        ,sigma/2], 

                       [0 , lambda_peak_1+cota_mean, np.inf, 1.1, np.inf ,sigma])
    
    
    p_predicted_core_E, p_cov_core_E = curve_fit(gauss_core_emission, x_datos, y_datos, p0=p0_core_emission, bounds=bounds_core_emission, max_nfev=50000)

    mse_core_E = mean_squared_error(y_linea,  gauss_core_emission(x_linea, *p_predicted_core_E))
    bic_core_E = calculate_bic(len(y_linea), mse_core_E, len(p_predicted_core_E))

    # MEJOR AJUSTE

    parameters =[p_opt_simple,p_opt_doble,p_predicted_core_E]
    gauss_func = [gauss,dos_gaussianas,gauss_core_emission]
    bic_values = [bic_simple,bic_doble,bic_core_E]
    names = ["Single Line","Blended Line", "Core Emission Line"]
    sigmas = [p_opt_simple[2],p_opt_doble[2],p_predicted_core_E[2]]
    p_lambda = [p_opt_simple[1],p_opt_doble[1],p_predicted_core_E[1]]
    idx = np.argmin(bic_values)

    # EW del mejor ajuste
    df_EW = df_continuo.loc[(df_continuo["lambda"]> p_lambda[idx] - 4*sigmas[idx]) &
                            (df_continuo["lambda"]< p_lambda[idx] + 4*sigmas[idx])].reset_index(drop=True)
    x_EW,y_EW = df_EW["lambda"].values,df_EW["flux"].values

    if (names[idx]== "Single Line") | (names[idx]== "Core Emission Line") :   
        continuo_linea =  coeffs[0] * x_EW + coeffs[1]
        EW4 =idl_tabulated(x_EW,1-(y_EW/continuo_linea))
    else:
        df_EW["gauss_1"] = gauss(x_EW, *np.concatenate((p_opt_doble[:3], p_opt_doble[-1]), axis=None))
        df_EW["gauss_2"] = gauss(x_EW, *p_opt_doble[3:])        
        EW4 = idl_tabulated(x_EW,1-(df_EW["gauss_1"].values))
        EW4_g2 =idl_tabulated(x_EW,1-(df_EW["gauss_2"].values))
        
    if plot==True:
        ax.axvline(x=p_lambda[idx] - 4*sigmas[idx], color='r', linestyle='--',label="-4 Sigma Best Gauss")
        ax.axvline(x=p_lambda[idx] + 4*sigmas[idx], color='r', linestyle='--',label="+4 Sigma Best Gauss")
        ax.axvline(x=df_EW["lambda"].min(), color='r', linestyle='dotted',label="-4 Sigma First Gauss")
        ax.axvline(x=df_EW["lambda"].max(), color='r', linestyle='dotted',label="+4 Sigma First Gauss")
        ax.set_title(f"{line_type}[{sp}]")
        ax.set_title(f" {np.round(EW4,2)} {bic_simple} [{sp}]")
        if names[idx] == "Blended Line":
            ax.plot(df_continuo["lambda"],df_continuo["flux_norm_line"],"o-",c="grey")
            ax.plot(df_EW["lambda"],df_EW["flux_norm_line"],"o-",c="black")
            ax.plot(df_EW["lambda"],df_EW["gauss_1"],c="red",linestyle="--")
            ax.plot(df_EW["lambda"],df_EW["gauss_2"],c="blue",linestyle="--")
        else:
            ax.plot(df_continuo["lambda"],df_continuo["flux"],"o-",c="grey")
            ax.plot(df_EW["lambda"],df_EW["flux"],"o-",c="black")
            ax.plot(df_continuo["lambda"],coeffs[0] * df_continuo["lambda"] + coeffs[1],c="green")
        ax.legend()
        return 
    
    return EW4

In [ ]:
def ajuste_HeIMgII(df_continuo,lambda_peak_1,lambda_peak_2,line_type,coeffs,spec_line,cota_mean,R,sp,plot=False,ax=None):
    # Extraer datos
    x_datos = df_continuo["lambda"].values
    y_datos = df_continuo["flux_norm_line"].values
    # Valores iniciales para los parámetros del ajuste
    sigma = lambda_peak_1/(R*2.634)
    b = df_continuo["flux_norm_line"].quantile(0.9)
    ##AJUSTE UNA GAUSSIANA

    amplitude_1,A_lim_sup_1,A_lim_inf_1 = stimate_parameters_line_type(lambda_peak_1,3,df_continuo,line_type)
    p0_simple = [amplitude_1, lambda_peak_1, sigma, b]
    bounds_simple = ([A_lim_inf_1, lambda_peak_1-cota_mean, sigma, 0], [A_lim_sup_1, lambda_peak_1+cota_mean,np.inf, 1.5])
    # Ajuste de una sola gaussiana
    try:
        p_opt_simple, p_cov_simple = curve_fit(gauss, x_datos, y_datos, p0=p0_simple, bounds=bounds_simple, max_nfev=10000)
    except ValueError as e:
        if str(e) == "`x0` is infeasible." :   
            return np.nan,np.nan
    df_linea = df_continuo.loc[(df_continuo["lambda"]> p_opt_simple[1] - 6*p_opt_simple[2]) & (df_continuo["lambda"]< p_opt_simple[1] + 6*p_opt_simple[2])]
    x_linea,y_linea = df_linea["lambda"].values,df_linea["flux"].values
    mse_simple = mean_squared_error(y_linea,  gauss(x_linea, *p_opt_simple))
    bic_simple = calculate_bic(len(y_linea), mse_simple, len(p_opt_simple))    
    # AJUSTE DOS GAUSSIANAS
    delta_lambda = p_opt_simple[1] - lambda_peak_1
    lambda_peak_2 = lambda_peak_2+delta_lambda
    amplitude_2,A_lim_sup_2,A_lim_inf_2 = stimate_parameters_line_type(lambda_peak_2,3,df_continuo,line_type) 
    p0_doble = [amplitude_1,lambda_peak_1, sigma, amplitude_2, lambda_peak_2, sigma, b]
    bounds_doble = ([-np.inf, lambda_peak_1-cota_mean, sigma, -np.inf, lambda_peak_2-cota_mean, sigma, 0], 
                    [0, lambda_peak_1 + cota_mean, np.inf, 0, lambda_peak_2+cota_mean, np.inf,1.5])
    try:
        p_opt_doble, p_cov_doble = curve_fit(dos_gaussianas, x_datos, y_datos, p0=p0_doble, bounds=bounds_doble, max_nfev=10000)
        mse_doble = mean_squared_error(y_linea,  dos_gaussianas(x_linea, *p_opt_doble))
        bic_doble = calculate_bic(len(y_linea), mse_doble, len(p_opt_doble))
    except ValueError as e:
        if str(e) == "`x0` is infeasible." :   
            return np.nan,np.nan
        # EW del mejor ajuste
    df_EW = df_continuo.loc[(df_continuo["lambda"]> p0_doble[1] - 4*p0_doble[2]) &
                            (df_continuo["lambda"]< p0_doble[4] + 4*p0_doble[5])].reset_index(drop=True)
    x_EW,y_EW = df_EW["lambda"].values,df_EW["flux"].values

    df_EW["gauss_1"] = gauss(x_EW, *np.concatenate((p_opt_doble[:3], p_opt_doble[-1]), axis=None))
    df_EW["gauss_2"] = gauss(x_EW, *p_opt_doble[3:])        
    EW4 = idl_tabulated(x_EW,1-(df_EW["gauss_1"].values))
    EW4_g2 =idl_tabulated(x_EW,1-(df_EW["gauss_2"].values))        
    if plot==True:
        ax.axvline(x=p0_doble[1] - 4*p0_doble[2], color='r', linestyle='--',label="-4 Sigma Best Gauss")
        ax.axvline(x=p0_doble[4] + 4*p0_doble[5], color='r', linestyle='--',label="+4 Sigma Best Gauss")
        ax.axvline(x=df_EW["lambda"].min(), color='r', linestyle='dotted',label="-4 Sigma First Gauss")
        ax.axvline(x=df_EW["lambda"].max(), color='r', linestyle='dotted',label="+4 Sigma First Gauss")
        ax.plot(df_continuo["lambda"],df_continuo["flux_norm_line"],"o-",c="grey")
        ax.plot(df_EW["lambda"],df_EW["flux_norm_line"],"o-",c="black")
        ax.plot(df_EW["lambda"],df_EW["gauss_1"],c="red",linestyle="--")
        ax.plot(df_EW["lambda"],df_EW["gauss_2"],c="blue",linestyle="--")
        ax.legend()
        return

        
    return EW4,EW4_g2

In [ ]:
def main_HeIMgII(filename,
                    function,
                    R,
                    rv,
                    sigma_R1300_transformed,
                    sp,
                    rms_lamost,
                    blue_cont,
                    red_cont,
                    spec_line,
                    lambda_peak,
                    threshold,
                    ax=None,
                    plot=False,
                ):

    spectra,coeffs,line_type_1 = Detect_line_static_and_var(filename,
                    function,
                    R,
                    rv,
                    sigma_R1300_transformed,
                    sp,  
                    rms_lamost,
                    blue_cont,
                    red_cont,
                    spec_line,
                    lambda_peak,
                    threshold,plot=False)
    
    linea_1 = 4471
    linea_2 = 4481
    if line_type_1!="U":
        EW_line1,EW_line2 = ajuste_HeIMgII(spectra,linea_1,linea_2,line_type_1,coeffs,spec_line,3,1300,sp)
    else:
        EW_line1,EW_line2 = np.nan,np.nan
    return EW_line1,EW_line2

In [ ]:
def main_function(filename,
                    function,
                    R,
                    rv,
                    sigma_R1300_transformed,
                    sp,
                    rms_lamost,
                    blue_cont,
                    red_cont,
                    spec_line,
                    lambda_peak,
                    threshold,
                    ax=None,
                    plot=False,
                ):

    spectra,coeffs,line_type_1 = Detect_line_static_and_var(filename,
                    function,
                    R,
                    rv,
                    sigma_R1300_transformed,
                    sp,  
                    rms_lamost,
                    blue_cont,
                    red_cont,
                    spec_line,
                    lambda_peak,
                    threshold,plot=False)
    
    linea_1 = lines[key]["central_lambda"]
    linea_2 = None
    if linea_1 == 4481:
        linea_2 =  4471
    if linea_1 == 4471:
        linea_2 = 4481
    if linea_1 ==4650:
        linea_2 = 4640 #NIII
    if linea_1 == 4088:
        linea_2 = 4102 #H delta
    if linea_1 ==4121:
        linea_2 =4102 # H delta
    if linea_1 == 4200:
        linea_2 = 4189 # OII
    if linea_1 == 4387:
        linea_2 = 4380 # NII
    if linea_1 ==4553:
        linea_2 = 4541 # He I

    if line_type_1!="U":
        try:
            EW = ajuste_gaussiano(spectra,linea_1,linea_2,line_type_1,coeffs,spec_line,3,1300,sp)
        except ValueError as e:
            if str(e) == "`x0` is infeasible." :
                line_type_1 = "U"
                EW = np.nan
    else:
        EW= np.nan
    return EW, line_type_1

In [ ]:
# Create a figure with 10 rows and 5 columns
for key in list(lines.keys()):
    fig, axes = plt.subplots(10, 5, figsize=(20, 40))
    
    # Flatten the axes array for easy iteration
    axes = axes.flatten()
    
    for i in range(50):
        Detect_line_static_and_var(df["fits"][i],
                    df["function"][i],
                    df["R"][i],
                    df["rv"][i],
                    df["sigma_R1300_transformed"][i],
                    df["SpT"][i],  
                    0,
                    lines[key]["continuum_blue"],
                    lines[key]["continuum_red"],
                    lines[key]["spectral_line"],
                    lines[key]["central_lambda"],
                    5,ax=axes[i],plot=True)
    plt.tight_layout()
    fig.savefig(f"{path_plots}/static_and_var/{key}.pdf")

In [ ]:
# Create a figure with 10 rows and 5 columns
for key in list(lines.keys()):
    fig, axes = plt.subplots(10, 5, figsize=(20, 40))
    
    # Flatten the axes array for easy iteration
    axes = axes.flatten()
    
    for i in range(50):
        Detect_line_static(df["fits"][i],
                    df["function"][i],
                    df["R"][i],
                    df["rv"][i],
                    df["sigma_R1300_transformed"][i],
                    df["SpT"][i],  
                    0,
                    lines[key]["continuum_blue"],
                    lines[key]["continuum_red"],
                    lines[key]["spectral_line"],
                    lines[key]["central_lambda"],
                    3,ax=axes[i],plot=True)
    plt.tight_layout()
    fig.savefig(f"{path_plots}/static_and_var/{key}.pdf")

In [ ]:
# Create a figure with 10 rows and 5 columns
for key in list(lines.keys()):
    fig, axes = plt.subplots(10, 5, figsize=(20, 40))
    
    # Flatten the axes array for easy iteration
    axes = axes.flatten()
    
    for i in range(50):
        Detect_line_bayo2011(df["fits"][i],
                    df["function"][i],
                    df["R"][i],
                    df["rv"][i],
                    df["sigma_R1300_transformed"][i],
                    df["SpT"][i],  
                    0,
                    lines[key]["central_lambda"],
                    3,ax=axes[i])
    plt.tight_layout()
    fig.savefig(f"{path_plots}/Bayo2011_var/{key}.pdf")

In [ ]:
np.random.seed(42)
for key in ['He I 4471', 'Mg II 4481', 'Fe II 4233', 'He II 4541', 'He II 4686']:  #list(lines.keys()):
    selection = np.random.choice(rms_values, len(df))
    df["Lamost_rms"] = selection  # Set the random seed for reproducibility
    fig, axes = plt.subplots(10, 5, figsize=(20, 40))
    
    # Flatten the axes array for easy iteration
    axes = axes.flatten()
    
    for i in range(50):
        spectra,coeffs,line_type_1 = Detect_line_static_and_var(df["fits"][i],
                    df["function"][i],
                    df["R"][i],
                    df["rv"][i],
                    df["sigma_R1300_transformed"][i],
                    df["SpT"][i],  
                    df["Lamost_rms"][i],
                    lines[key]["continuum_blue"],
                    lines[key]["continuum_red"],
                    lines[key]["spectral_line"],
                    lines[key]["central_lambda"],
                    5)
        linea_1 = lines[key]["central_lambda"]
        linea_2 = None
        if linea_1 == 4481:
            linea_2 = 4471
        if linea_1 == 4471:
            linea_2 = 4481
        if linea_1 ==4650:
            linea_2 = 4640 #NIII
        if linea_1 == 4088:
            linea_2 = 4102 #H delta
        if linea_1 ==4121:
            linea_2 =4102 # H delta
        if linea_1 == 4200:
            linea_2 = 4189 # OII
        if linea_1 == 4387:
            linea_2 = 4380 # NII
        if linea_1 ==4553:
            linea_2 = 4541 # He II
        if line_type_1!="U":
            try:
                ajuste_gaussiano(spectra,linea_1,linea_2,line_type_1,coeffs,lines[key]["spectral_line"],3,1300,df["SpT"][i],True,axes[i])
            except ValueError as e:
                if (str(e) == "`x0` is infeasible." ) | (line_type_1=="U"):
                    axes[i].plot(spectra["lambda"],spectra["flux"],"o-",c="black")
                    axes[i].set_title("`x0` is infeasible.")
        else:
            axes[i].plot(spectra["lambda"],spectra["flux"],"o-",c="black")
            axes[i].set_title("U")
    plt.tight_layout()
    fig.savefig(f"{path_plots}/gauss/{key}.pdf")


In [ ]:
np.random.seed(1)  # Set the random seed for reproducibility
for key in list(lines.keys()):
    selection = np.random.choice(rms_values, len(df))
    df["Lamost_rms"] = selection
    detection = f'{key}_type'
    EW_name = f'{key}_EW'
    
    df[[EW_name,detection]] = df.parallel_apply(
            lambda row: main_function(
            row["fits"],
            row["function"],
            row["R"],
            row["rv"],
            row["sigma_R1300_transformed"],
            row["SpT"],
            0,
            lines[key]["continuum_blue"],
            lines[key]["continuum_red"],
            lines[key]["spectral_line"],
            lines[key]["central_lambda"],
            3
        ),
        axis=1,
        result_type="expand")

In [ ]:
key = "He I 4471"
df[["THeI4471","TMgII4481"]]= df.parallel_apply(
            lambda row: main_HeIMgII(
                    row["fits"],
                    row["function"],
                    row["R"],
                    row["rv"],
                    row["sigma_R1300_transformed"],
                    row["SpT"],  
                    0,
                    lines[key]["continuum_blue"],
                    lines[key]["continuum_red"],
                    lines[key]["spectral_line"],
                    lines[key]["central_lambda"],
                    5),
        axis=1,result_type="expand")

In [ ]:
teoric_He_over_Mg = df["THeI4471"]/ df["TMgII4481"]

In [ ]:
df["sp"] = np.nan
df["THeMg"] = df["THeI4471"]/ df["TMgII4481"]

In [ ]:
df["EHeMg"] = df['He I 4471_EW'] / df['Mg II 4481_EW']

In [ ]:
df.loc[df["THeMg"].isna(),"THeMg"] = df.loc[df["THeMg"].isna()]["He I 4471_EW"] / df.loc[df["THeMg"].isna()]["Mg II 4481_EW"]

In [ ]:
lamost = df 

In [ ]:
lamost.loc[((lamost["He II 4686_type"]=="abs")&(lamost["He II 4541_type"]=="abs")),"sp"] = "O-B0"
lamost.loc[(lamost["He I 4471_type"]=="U")&
                    (lamost["Mg II 4481_type"]!="U") &
                    (lamost["sp"].isna()),"sp"] = "Later"
lamost.loc[(lamost["He I 4471_type"]!="U")&
                    (lamost["Mg II 4481_type"]=="U") &
                    (lamost["sp"].isna()),"sp"] = "B0-B3"
lamost.loc[(lamost["He I 4471_type"]!="U")&
            (lamost["Mg II 4481_type"]=="U")&
            (lamost["He I 4144_type"]=="U")&
                (lamost["sp"].isna()),"sp"] = "B4-B7"
lamost.loc[(lamost["EHeMg"] >= 3)&
                    (lamost["He I 4144_type"]=="abs") &
                    (lamost["sp"].isna()),"sp"] = "B0-B3"
lamost.loc[(lamost["EHeMg"] > 1)&
                    (lamost["He I 4144_type"]=="abs") &
                    (lamost["sp"].isna()),"sp"] = "B4-B7"
lamost.loc[(lamost["EHeMg"] <= 1)&
                    (lamost["He I 4144_type"]=="abs") &
                    (lamost["sp"].isna()),"sp"] = "B8-A1"
lamost.loc[(lamost["EHeMg"] > 1)&
                    (lamost["He I+He II 4026_type"]=="abs") &
                   ( (lamost["Fe II 4233_type"]=="U")|(lamost["Fe II 4233_EW"]<=0.15))&
                    (lamost["sp"].isna()),"sp"] = "B4-B7"
lamost.loc[(lamost["EHeMg"] <= 1)&
                    (lamost["He I+He II 4026_type"]=="abs") &
                     ( (lamost["Fe II 4233_type"]=="U")|(lamost["Fe II 4233_EW"]<=0.15))&
                    (lamost["sp"].isna()),"sp"] = "B8-A1"

lamost.loc[(lamost["He I 4144_type"]!="abs") &
                    (lamost["He I+He II 4026_type"]!="abs") &
                    (lamost["sp"].isna()),"sp"] = "Later"

lamost.loc[(lamost["He I 4144_type"]!="abs") &
                    (lamost["He I+He II 4026_type"]=="abs") &
                    (lamost["Fe II 4233_EW"]>0.15)&
                    (lamost["sp"].isna()),"sp"] = "Later"

lamost.loc[(lamost["EHeMg"] <= 1)&
                    (lamost["He I 4144_type"]=="abs") &
                    (lamost["sp"].isna()),"sp"] = "B8-A1"

In [ ]:
%ls $path

In [ ]:
lamost[["Name","sp","SpT",
    "He II 4686_type",
    "He II 4541_type",
    "He I 4471_type",
    "Mg II 4481_type",
    "He I 4144_type",
    "THeI4471",
    "He I 4471_EW",    
    "TMgII4481",
    "Mg II 4481_EW",
    "He I+He II 4026_type",
    "Fe II 4233_type",
    "Fe II 4233_EW"
]].to_csv(f"{path}Table_Standard_Classificated.csv",index=False)

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Primer gráfico
axs[0].plot(lamost["Mg II 4481_EW"], lamost["TMgII4481"], "k.")
axs[0].set_xlabel("Teoric EW")
axs[0].set_ylabel("Empirically EW")
axs[0].set_title("Mg II 4481")

# Segundo gráfico
axs[1].plot(lamost["He I 4471_EW"], lamost["THeI4471"], "k.")
axs[1].set_xlabel("Teoric EW")
axs[1].set_ylabel("Empirically EW")
axs[1].set_title("He I 4471")

plt.tight_layout()
plt.show()


In [ ]:
df.groupby(["sp","SpT"]).count()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Definir los colores base para cada letra
base_colors = {
    'O': sns.color_palette("Blues", 10),
    'B': sns.color_palette("YlOrBr", 10),
    'A': sns.color_palette("Greens", 2)
}

# Crear un diccionario de colores para cada tipo espectral
color_map = {}
for sp in df["SpT"].unique():
    base_color = base_colors[sp[0]]
    index = int(sp[1])  # Usar el número directamente como índice
    color_map[sp] = base_color[index % len(base_color)]  # Asegurar que el índice esté dentro del rango

# Graficar
sns.scatterplot(data=df, x="Mg II 4481_EW", y="He I 4471_EW", hue="SpT",style="He I 4144_type", palette=color_map)
x = np.linspace(0, 0.6, 100)
y = x
plt.plot(x, 3*y, label='B3', color='black', linestyle='--')
plt.plot(x, y, label='B6-B7', color='black', linestyle='-')
plt.legend()
plt.show()


In [ ]:
df["sp_round"] = df["SpT"].str[0:2]

In [ ]:
df.loc[df["sp_round"].str[0]=="O","sp_round"] = "O"

In [ ]:
np.random.seed(1)  # Set the random seed for reproducibility
valores_matriz = np.zeros((20,11,18))
for i in range(20):
    selection = np.random.choice(rms_values, len(df))
    df["Lamost_rms"] = selection
    for key in list(lines.keys()):
        detection = f'{key}_type'
        EW_name = f'{key}_EW'
        
        df[[EW_name,detection]] = df.parallel_apply(
                lambda row: main_function(
                row["fits"],
                row["function"],
                row["R"],
                row["rv"],
                row["sigma_R1300_transformed"],
                row["SpT"],
                row["Lamost_rms"],
                lines[key]["continuum_blue"],
                lines[key]["continuum_red"],
                lines[key]["spectral_line"],
                lines[key]["central_lambda"],
                5
            ),
            axis=1,
            result_type="expand")
            

    sp = "sp_round"

    spectral_lines = [col for col in df.columns if '_type' in col]
    
    counts = {}
    
    # Iterar sobre cada línea espectral
    for line in list(lines.keys()):
        line = str(line)+"_type"
        df_variable = df.loc[df[line]!="error"].reset_index(drop=True)
        # Crear una Serie que cuenta las ocurrencias de 'U' por cada 'SpT'
        count = df_variable[df_variable[line] == 'U'].groupby(sp)[line].count()
        total_count = df_variable.groupby(sp)[line].count()
        percentage = (count / total_count)
        percentage = np.round(percentage,2)# Calcula el porcentaje
        counts[line] = percentage
    
    # Convertir el diccionario en un DataFrame
    result_df = pd.DataFrame(counts)
    
    # Rellenar los NaN con 0 para mostrar ausencia de 'U'
    
    result_df.rename(columns={f"{value}_type": f"key{index}" for index, value in enumerate(lines)}, inplace=True)
    
    new_order = ['O', 'B0', 'B1', 'B2', 'B3', 'B5', 'B7', 'B8', 'B9', 'A0', 'A1']
    
    # Reordenar las filas usando reindex
    result_df = result_df.reindex(new_order)
    result_df = result_df.fillna(0)
    valores_matriz[i] = result_df.values


In [ ]:
# Calcular media y desviación estándar sobre el eje 0
mean_values = valores_matriz.mean(axis=0).round(2)
std_values = valores_matriz.std(axis=0).round(2)

# Continúa utilizando mean_values y std_values como arrays numpy
data_for_heatmap = mean_values  # Usar la media para el mapa de calor

# Convertir a DataFrame para usar strings como anotaciones
df_for_heatmap = pd.DataFrame(data_for_heatmap)
annot = [[f"{m}\n± {s}" for m, s in zip(mean_row, std_row)]
         for mean_row, std_row in zip(mean_values, std_values)]

df_for_heatmap.columns = df_for_heatmap.columns = list(lines.keys()) # Ajusta columnas si es necesario
df_for_heatmap.index = result_df.index[:df_for_heatmap.shape[0]] 

# Crear el mapa de calor con anotaciones personalizadas
plt.figure(figsize=(15, 10))
sns.heatmap(df_for_heatmap, annot=annot, fmt="", cmap='Blues', vmin=0, vmax=1)
plt.title('Percentage of non-detections')
plt.ylabel("")
plt.xticks(rotation=45, fontsize=12)  # Rotar y cambiar tamaño de labels del eje x
plt.yticks(fontsize=12)
plt.savefig(f"{path_plots}CM_lamost_RMS.pdf", format='pdf', bbox_inches='tight')
plt.show()
